# RoboFactory

> RoboFactory tasks (multi-arm manipulation on ManiSkill) as stable-marl environments, with per-arm cameras and joint states.

In [ ]:
#| default_exp envs.robofactory.env

In [ ]:
#| hide
from nbdev.showdoc import *

[RoboFactory](https://github.com/MARS-EAI/RoboFactory) (Qin et al., 2025): multi-arm manipulation tasks
built on [ManiSkill 3](https://maniskill.readthedocs.io), where 2 to 4 Panda arms have to work together.
They are also the manipulation environments of MangoBench (its tasks and goal-conditioned baselines, see
`stable_marl.wm.gcrl`). RoboFactory's tasks, scenes and motion-planning experts are vendored in
`stable_marl/envs/robofactory/upstream/` (MIT); ManiSkill is an optional dependency
(`pip install 'stable-marl[robofactory]'`), imported only when an env is created, and the 3D assets
(40 MB) are downloaded on first use (`download_assets`).

| task | arms | goal |
|---|---|---|
| `LiftBarrier` | 2 | lift a barrier together, one arm at each end |
| `PassShoe` | 2 | one arm hands a shoe to the other, which puts it down |
| `PlaceFood` | 2 | one arm opens the pot, the other puts the food in |
| `TwoRobotsStackCube` | 2 | stack two cubes, each out of reach of one arm |
| `CameraAlignment` | 3 | two arms hold up objects, the third aligns a camera with them |
| `ThreeRobotsStackCube` | 3 | stack three cubes |
| `TakePhoto` | 4 | hold up an object and press a camera's button over it |
| `LongPipelineDelivery` | 4 | pass an object along the four arms |

`RoboFactoryEnv` runs one task (one ManiSkill env, on the CPU simulation; a `World`'s pool holds several)
as a stable-marl `MultiAgentEnv`:

| key | per arm | content |
|---|---|---|
| `pixels` | `(H, W, 3)` uint8 | the arm's own camera (RoboFactory's `head_camera_agent<i>`, 240 x 320 unless `camera_size`) |
| `vector` | `(18,)` float32 | the arm's joint positions (7 + 2 fingers), then velocities |
| `success` (info) | bool | the task is done |

Actions are continuous, RoboFactory's `pd_joint_pos`: the 7 joint targets and the gripper
(-1 closed, 1 open) of each arm. The reward is ManiSkill's sparse one, 1 when the task is done
(RoboFactory's dense rewards are all 0), shared by the arms, and the end of episode applies to every arm:
termination on success, truncation at `max_steps` (the task's limit by default, 500 for `LiftBarrier`).
`state()` is the simulation state (poses and velocities of every object and arm, ManiSkill's
`get_state()`), which also restores recorded steps in dataset-driven evaluation. `render()` is the
global camera of the scene (`render_size`).

In [ ]:
#| export
from __future__ import annotations

import os
from typing import Any

import numpy as np
from gymnasium import spaces

from stable_marl.envs.base import MultiAgentEnv
from stable_marl.types import AgentID

In [ ]:
#| export
TASKS = {   # task: (RoboFactory config file, number of arms)
    'LiftBarrier': ('lift_barrier', 2),
    'PassShoe': ('pass_shoe', 2),
    'PlaceFood': ('place_food', 2),
    'TwoRobotsStackCube': ('two_robots_stack_cube', 2),
    'CameraAlignment': ('camera_alignment', 3),
    'ThreeRobotsStackCube': ('three_robots_stack_cube', 3),
    'TakePhoto': ('take_photo', 4),
    'LongPipelineDelivery': ('long_pipeline_delivery', 4),
}
SCENES = ('table', 'robocasa')
ASSET_REPO = 'sparklexfantasy/RoboFactory_asset'   # RoboFactory's 3D assets (Hugging Face dataset)


def asset_dir() -> str:
    "Where RoboFactory's assets live: ``$ROBOFACTORY_ASSET_DIR``, else ``<cache_dir>/robofactory/assets``."
    return os.path.expanduser(os.environ.get('ROBOFACTORY_ASSET_DIR') or os.path.join(
        os.environ.get('STABLEMARL_HOME', '~/.stable_marl'), 'robofactory', 'assets'))


def download_assets(robocasa: bool = False) -> str:
    """
    Download RoboFactory's 3D assets (objects and table, 40 MB) to :func:`asset_dir`, and with
    `robocasa` ManiSkill's RoboCasa kitchens (for ``scene='robocasa'``, several GB, to ManiSkill's
    asset folder). Done automatically by the first `RoboFactoryEnv`; call it beforehand on machines
    without internet access at run time (e.g. HPC compute nodes).
    """
    from huggingface_hub import snapshot_download
    path = asset_dir()
    snapshot_download(repo_id=ASSET_REPO, repo_type='dataset', local_dir=path, ignore_patterns=['assets.zip'])
    if robocasa:
        import subprocess, sys
        subprocess.run([sys.executable, '-m', 'mani_skill.utils.download_asset', 'RoboCasa', '-y'], check=True)
    return path

In [ ]:
#| export
def _numpy(x) -> np.ndarray:
    "ManiSkill's batched tensors (one env) as numpy, without the batch dimension."
    return (x.detach().cpu().numpy() if hasattr(x, 'detach') else np.asarray(x))[0]


class RoboFactoryEnv(MultiAgentEnv):
    """
    The RoboFactory `task` (one of :data:`TASKS`) as a stable-marl env.

    Parameters
    ----------
    task : str
        RoboFactory task, e.g. 'LiftBarrier', 'TakePhoto'
    scene : str
        'table' (default) or 'robocasa' (a kitchen; needs ``download_assets(robocasa=True)``)
    observations : tuple
        Per-arm observations: 'pixels' (the arm's camera) and / or 'vector' (joint positions and velocities)
    camera_size : tuple, optional
        (height, width) of the arms' cameras; default RoboFactory's (240, 320)
    max_steps : int, optional
        Step limit (truncation); default the task's
    render_size : tuple
        (height, width) of :meth:`render`, the global camera
    reward_mode : str
        ManiSkill's reward: 'sparse' (1 on success) or 'none'
    shader : str
        ManiSkill's shader of the cameras: 'minimal' (fastest), 'default', 'rt' (ray tracing)
    """
    def __init__(self, task: str = 'LiftBarrier', scene: str = 'table', observations: tuple = ('pixels', 'vector'),
                 camera_size: tuple | None = None, max_steps: int | None = None, render_size: tuple = (240, 320),
                 reward_mode: str = 'sparse', shader: str = 'default', render_mode: str | None = 'rgb_array'):
        if task not in TASKS:
            raise ValueError(f"unknown RoboFactory task {task!r}: one of {list(TASKS)}")
        if scene not in SCENES:
            raise ValueError(f"unknown scene {scene!r}: one of {SCENES}")
        observations = tuple(observations)
        if not observations or set(observations) - {'pixels', 'vector'}:
            raise ValueError(f"observations: 'pixels' and / or 'vector', not {observations}")
        if not os.path.isdir(os.path.join(asset_dir(), 'objects')):
            print(f"Downloading RoboFactory's assets to {asset_dir()}")
            download_assets()
        import gymnasium as gym
        from stable_marl.envs.robofactory import upstream
        self.task, self.scene, self.observations = task, scene, observations
        self.num_agents = TASKS[task][1]
        self.render_mode = render_mode
        self.config = os.path.join(upstream.CONFIG_DIR, scene, f'{TASKS[task][0]}.yaml')
        cameras = dict(shader_pack=shader)
        if camera_size is not None:
            cameras.update(height=int(camera_size[0]), width=int(camera_size[1]))
        kwargs = dict(max_episode_steps=max_steps) if max_steps is not None else {}
        self._env = gym.make(f'{task}-rf', config=self.config, obs_mode='rgb' if 'pixels' in observations else 'state_dict',
                             control_mode='pd_joint_pos', reward_mode=reward_mode, render_mode='rgb_array',
                             sim_backend='cpu', sensor_configs=cameras,
                             human_render_camera_configs=dict(height=int(render_size[0]), width=int(render_size[1]),
                                                              shader_pack=shader), **kwargs)
        self.max_steps = getattr(self._env, '_max_episode_steps', None)    # ManiSkill's TimeLimitWrapper
        self.agent_names = list(self._env.unwrapped.action_space.keys())      # 'panda-0', 'panda-1', ...
        self._obs = self._last_info = None
        self._episode = 0
        self.observation_space = spaces.Dict({a: spaces.Dict(self._agent_space()) for a in range(self.num_agents)})
        self.action_space = spaces.Dict({a: spaces.Box(np.asarray(s.low, np.float32), np.asarray(s.high, np.float32), dtype=np.float32)
                                         for a, s in enumerate(self._env.unwrapped.action_space.values())})

    def _agent_space(self) -> dict:
        out = {}
        if 'pixels' in self.observations:
            cam = self._env.unwrapped.observation_space['sensor_data']['head_camera_agent0']['rgb']
            out['pixels'] = spaces.Box(0, 255, cam.shape[-3:], np.uint8)
        if 'vector' in self.observations:
            out['vector'] = spaces.Box(-np.inf, np.inf, (18,), np.float32)
        return out

    @property
    def state_space(self) -> spaces.Space:
        "ManiSkill's simulation state: poses and velocities of every object, joint states of every arm."
        return spaces.Box(-np.inf, np.inf, (int(self._env.unwrapped.get_state().shape[-1]),), np.float32)

    def state(self) -> np.ndarray:
        return _numpy(self._env.unwrapped.get_state()).astype(np.float32)

    def _observations(self, obs) -> dict[AgentID, dict]:
        out = {}
        for a, name in enumerate(self.agent_names):
            out[a] = {}
            if 'pixels' in self.observations:
                out[a]['pixels'] = _numpy(obs['sensor_data'][f'head_camera_agent{a}']['rgb'])
            if 'vector' in self.observations:
                joints = obs['agent'][name]
                out[a]['vector'] = np.concatenate([_numpy(joints['qpos']), _numpy(joints['qvel'])]).astype(np.float32)
        return out

    def _infos(self, info) -> dict[AgentID, dict]:
        success = bool(_numpy(info['success'])) if 'success' in info else False
        return {a: {'success': success} for a in range(self.num_agents)}

    def reset(self, seed: int | None = None, options: dict | None = None):
        """
        `options`: ``{'sim_state': state}`` starts from a recorded :meth:`state` (after the reset of `seed`).
        RoboFactory places the objects with numpy's global random generator, so it is seeded here too
        (and restored afterwards): a seed gives the same scene every time.
        """
        import torch
        super().reset(seed=seed)
        seed = int(self.np_random.integers(2 ** 31)) if seed is None else int(seed)
        rng_state = np.random.get_state()
        np.random.seed(seed)
        try:
            obs, info = self._env.reset(seed=seed)
        finally:
            np.random.set_state(rng_state)
        if options and options.get('sim_state') is not None:
            base = self._env.unwrapped
            base.set_state(torch.as_tensor(np.asarray(options['sim_state']), dtype=torch.float32)[None])
            obs, info = base.get_obs(), base.get_info()
        self._episode += 1
        self._seed = seed
        self._obs, self._last_info = obs, info
        return self._observations(obs), self._infos(info)

    def step(self, actions: dict[AgentID, Any]):
        act = {name: np.asarray(actions[a], np.float32).reshape(-1) for a, name in enumerate(self.agent_names)}
        obs, reward, terminated, truncated, info = self._env.step(act)
        self._obs, self._last_info = obs, info
        r, term, trunc = float(_numpy(reward)), bool(_numpy(terminated)), bool(_numpy(truncated))
        agents = range(self.num_agents)
        return (self._observations(obs), {a: r for a in agents}, {a: term for a in agents},
                {a: trunc for a in agents}, self._infos(info))

    def hold_action(self) -> np.ndarray:
        "Actions keeping every arm where it is, (num_agents, 8): its joint positions, the gripper's opening."
        out = []
        for agent in self._env.unwrapped.agent.agents:
            q = _numpy(agent.robot.get_qpos())
            low, high = -0.01, 0.04                              # pd_joint_pos's gripper range (Panda)
            out.append(np.r_[q[:7], 2 * (q[7] - low) / (high - low) - 1].astype(np.float32))
        return np.stack(out)

    def render(self) -> np.ndarray:
        "The global camera, (render_size, 3) uint8."
        return _numpy(self._env.render())

    def reset_options_from_dataset(self, init_row: dict, goal_row: dict) -> dict:
        "Reset options restoring the recorded step `init_row` (its ``state``); the goal is the task's success."
        return {'sim_state': np.asarray(init_row['state'])}

    def close(self):
        self._env.close()

### Tests

In [ ]:
# imports on their own: the docs build runs the cells with imports, and the tests need ManiSkill
import tempfile
from fastcore.test import test_fail
from stable_marl import World, RandomPolicy
from stable_marl.data import HDF5Dataset

In [ ]:
env = RoboFactoryEnv('LiftBarrier', camera_size=(64, 64), max_steps=20, render_size=(48, 64))
obs, infos = env.reset(seed=0)
assert env.num_agents == 2 and env.agent_names == ['panda-0', 'panda-1'] and env.max_steps == 20
assert obs[0]['pixels'].shape == (64, 64, 3) and obs[0]['pixels'].dtype == np.uint8 and obs[1]['vector'].shape == (18,)
assert env.observation_space[0].contains(obs[0]) and infos[0] == {'success': False}
assert env.action_space[1].shape == (8,) and env.state().shape == env.state_space.shape
assert env.render().shape == (48, 64, 3)
# seeded: the same scene for the same seed (RoboFactory randomizes with numpy's global generator)
s0 = env.state()
np.random.seed(123); before = np.random.rand()
env.reset(seed=0)
assert np.allclose(env.state(), s0)
np.random.seed(123); assert np.random.rand() == before                         # global generator left as it was
env.reset(seed=1)
assert not np.allclose(env.state(), s0)
# the arms hold still when commanded their current joints; episodes end at max_steps for every arm
env.reset(seed=0)
hold = env.hold_action()
assert hold.shape == (2, 8) and np.allclose(hold[:, :7], [o['vector'][:7] for o in obs.values()], atol=1e-5)
for t in range(20):
    obs, rew, term, trunc, info = env.step(dict(enumerate(hold)))
assert all(trunc.values()) and not any(term.values()) and rew == {0: 0.0, 1: 0.0}
assert np.allclose(env.state()[:20], s0[:20], atol=1e-2)
# restoring a recorded state
mid = env.state()
env.reset(seed=0, options=env.reset_options_from_dataset({'state': mid}, {}))
assert np.allclose(env.state(), mid, atol=1e-5)
# vector-only observations; argument checks
env = RoboFactoryEnv('TakePhoto', observations=('vector',))
obs, _ = env.reset(seed=0)
assert env.num_agents == 4 and set(obs[3]) == {'vector'}
test_fail(lambda: RoboFactoryEnv('Lift'), contains='unknown RoboFactory task')
test_fail(lambda: RoboFactoryEnv(observations=('depth',)), contains="'pixels' and / or 'vector'")

In [ ]:
# every task runs in a World: datasets with both observations, evaluation, dataset-driven evaluation
with tempfile.TemporaryDirectory() as tmp:
    for task, (_, arms) in TASKS.items():
        world = World(f'RoboFactory-{task}-v0', num_envs=2, max_episode_steps=4, camera_size=(32, 32))
        world.set_policy(RandomPolicy(seed=0))
        world.collect(f'{tmp}/{task}.h5', episodes=2, seed=0, progress=False)
        ds = HDF5Dataset(path=f'{tmp}/{task}.h5', num_steps=3)
        assert ds[0]['pixels'].shape == (3, arms, 32, 32, 3) and ds[0]['action'].shape == (3, arms * 8), task
        assert len(world.evaluate(episodes=2, seed=1)['episode_returns']) == 2
    world = World('RoboFactory-LiftBarrier-v0', num_envs=2, max_episode_steps=4, camera_size=(32, 32))
    world.set_policy(RandomPolicy(seed=0))
    ds = HDF5Dataset(path=f'{tmp}/LiftBarrier.h5', num_steps=1)
    res = world.evaluate(dataset=ds, episodes_idx=[0, 1], start_steps=[1, 0], goal_offset=2, eval_budget=3)
    assert res['episode_lengths'].tolist() == [3, 3] and 'goal' in world.infos

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()